In [1]:
import pandas as pd
import numpy as np
import optuna
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [2]:
# Cargar el dataset
df = pd.read_csv('../Datasets/machine_failure_dataset.csv')
df = pd.get_dummies(df, columns=['Machine_Type']) 
df.head()

,Temperature,Vibration,Power_Usage,Humidity,Failure_Risk,Machine_Type_Drill,Machine_Type_Lathe,Machine_Type_Mill
0,74.967142,56.996777,8.649643,20.460962,1,False,False,True
1,68.617357,54.623168,9.710963,25.698075,0,False,True,False
2,76.476885,50.298152,8.415160,27.931972,1,True,False,False
3,85.230299,46.765316,9.384077,39.438438,1,False,True,False
4,67.658466,53.491117,6.212771,32.782766,1,True,False,False


In [3]:
# Contar la cantidad de cada calse
df['Failure_Risk'].value_counts()

Failure_Risk
0    700
1    300
Name: count, dtype: int64

In [4]:
X = df.drop('Failure_Risk', axis=1).values
y = df['Failure_Risk'].values.reshape(-1, 1)
print(X.shape, y.shape)

(1000, 7) (1000, 1)


In [5]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
def objective(trial):
    try:
        # Definimos los espacios de búsqueda
        n_layers = trial.suggest_int('n_layers', 1, 5)
        activation = trial.suggest_categorical('activation', ['relu', 'gelu', 'elu'])
        lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'rmsprop'])
        
        model = Sequential()
        model.add(Input(shape=(X_train.shape[1],)))
        
        for i in range(n_layers):
            num_units = trial.suggest_int(f'n_units_l{i}', 8, 128)
            model.add(Dense(num_units, activation=activation))
            model.add(Dropout(trial.suggest_float(f'dropout_l{i}', 0.1, 0.3)))
        
        model.add(Dense(1, activation='sigmoid'))

        # SELECCIÓN DE OPTIMIZADOR SEGURA
        if optimizer_name == 'adam':
            optimizer = tf.keras.optimizers.Adam(learning_rate=lr)    
        else:
            optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)

        model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

        # Entrenamiento corto para la búsqueda
        model.fit(X_train, 
                  y_train, 
                  epochs=50, 
                  batch_size=32, 
                  verbose=0
                )   
        # Evaluar
        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        return accuracy

    except Exception as e:
        print(f"La prueba falló debido a: {e}")
        return 0.0

In [7]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) 

[I 2026-04-20 11:10:29,306] A new study created in memory with name: no-name-b07d5a73-90fb-44a0-bb4c-0e11d4bcc2e1
[I 2026-04-20 11:10:41,510] Trial 0 finished with value: 0.675000011920929 and parameters: {'n_layers': 4, 'activation': 'relu', 'learning_rate': 0.0014137092290073862, 'optimizer': 'rmsprop', 'n_units_l0': 34, 'dropout_l0': 0.273959526278241, 'n_units_l1': 74, 'dropout_l1': 0.22180487515036945, 'n_units_l2': 56, 'dropout_l2': 0.18293692415210194, 'n_units_l3': 36, 'dropout_l3': 0.2024508452587824}. Best is trial 0 with value: 0.675000011920929.
[I 2026-04-20 11:10:55,754] Trial 1 finished with value: 0.675000011920929 and parameters: {'n_layers': 1, 'activation': 'gelu', 'learning_rate': 0.00018342693884935705, 'optimizer': 'adam', 'n_units_l0': 77, 'dropout_l0': 0.294023799249032}. Best is trial 0 with value: 0.675000011920929.
[I 2026-04-20 11:11:10,517] Trial 2 finished with value: 0.675000011920929 and parameters: {'n_layers': 4, 'activation': 'gelu', 'learning_rate': 

In [8]:
print(f"Mejor Accuracy: {study.best_value:.4f}")
print("Mejores parámetros:", study.best_params)

Mejor Accuracy: 0.6750
Mejores parámetros: {'n_layers': 4, 'activation': 'relu', 'learning_rate': 0.0014137092290073862, 'optimizer': 'rmsprop', 'n_units_l0': 34, 'dropout_l0': 0.273959526278241, 'n_units_l1': 74, 'dropout_l1': 0.22180487515036945, 'n_units_l2': 56, 'dropout_l2': 0.18293692415210194, 'n_units_l3': 36, 'dropout_l3': 0.2024508452587824}


In [9]:
# Visualizar la importancia de los parámetros
optuna.visualization.plot_param_importances(study).show()
# Visualizar el historial de optimización
optuna.visualization.plot_optimization_history(study).show()